# 03 · Síntesis para el Visualizer

**Objetivo.** Ejecutar el pipeline del Analyzer, validar el contrato, escribir
el documento JSON y exportar los datasets a CSV y las figuras a PNG.

**Entrada.** `INPUT_PATH` (por defecto `results.json` si existe; si no,
`results-vuln.json`).

**Salida.**
- `analysis/outputs/analyzer_output.json`
- `analysis/outputs/csv/*.csv`
- `analysis/outputs/figures/*.png`

**Supuestos.** La lógica vive en `analysis.pipeline.run_analysis`; este notebook
solo orquesta, valida y exporta. Todo el contenido calculado es determinista;
`meta.generated_at` refleja el momento de ejecución y puede fijarse con
`GENERATED_AT`.

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
INPUT_PATH = REPO_ROOT / (
    "results.json" if (REPO_ROOT / "results.json").is_file() else "results-vuln.json"
)
OUTPUT_DIR = REPO_ROOT / "analysis" / "outputs"
OUTPUT_PATH = OUTPUT_DIR / "analyzer_output.json"
CSV_DIR = OUTPUT_DIR / "csv"
FIGURES_DIR = OUTPUT_DIR / "figures"

# `None` usa la marca de tiempo real; una cadena ISO-8601 fija la reproducibilidad.
GENERATED_AT = None

print("INPUT_PATH :", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)

In [ ]:
import sys

import matplotlib

matplotlib.use("Agg")  # backend no interactivo (sin display)
import matplotlib.pyplot as plt
import pandas as pd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import contract
from analysis.pipeline import run_analysis

## 1. Ejecutar el pipeline

`run_analysis` encadena `load_report → to_records → metrics → contract` y
escribe el documento validado.

In [ ]:
document = run_analysis(INPUT_PATH, OUTPUT_PATH, generated_at=GENERATED_AT)
errors = contract.validate_document(document)

print("schema_version       :", document["schema_version"])
print("organización         :", document["meta"]["organization"])
print("origen               :", document["meta"]["source_kind"])
print("repositorios         :", document["meta"]["repositories"])
print("generado             :", document["meta"]["generated_at"])
print("errores de validación:", errors)

assert not errors, errors
assert OUTPUT_PATH.is_file(), OUTPUT_PATH
print("Documento escrito en :", OUTPUT_PATH)

## 2. Exportar datasets a CSV

Cada dataset tabular se exporta a un CSV independiente en `analysis/outputs/csv/`.
Los objetos `concentration` y `relations` se aplanan en archivos propios.

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
written = []

# Columnas mínimas para que un dataset vacío conserve cabecera y siga siendo
# autocontenido (mismo esquema que el contrato).
FALLBACK_COLUMNS = {
    "repositories": [
        "repo", "full_name", "url", "commit", "status", "languages",
        "sbom_status", "sbom_components", "vuln_status", "vuln_total",
        "findings_total",
    ],
    "findings": ["repo", "rule_id", "severity", "file", "start_line"],
    "vulnerabilities": [
        "repo", "id", "severity", "package", "version", "type",
        "fixed_version", "namespace",
    ],
    "severity_distribution": ["severity", "count"],
    "severity_by_repo": ["repo", "severity", "count"],
    "top_rules": ["rule_id", "count", "repos_affected"],
    "top_cves": ["id", "severity", "count", "repos_affected"],
    "top_packages": ["package", "count", "repos_affected", "worst_severity"],
    "repository_distribution": [
        "repo", "vulnerabilities", "findings", "components", "status",
    ],
    "severity_by_package_type": ["type", "severity", "count"],
    "findings_by_language": ["language", "rule_id", "count"],
}


def _frame(rows, name: str) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    if frame.empty and not frame.columns.size:
        frame = pd.DataFrame(columns=FALLBACK_COLUMNS.get(name, []))
    return frame


for name in contract.LIST_DATASETS:
    path = CSV_DIR / f"{name}.csv"
    _frame(document["datasets"][name], name).to_csv(path, index=False)
    written.append(path.name)

concentration = document["datasets"]["concentration"]
pd.DataFrame([concentration]).to_csv(CSV_DIR / "concentration.csv", index=False)
written.append("concentration.csv")

relations = document["datasets"]["relations"]
pd.DataFrame(
    [
        {
            "pearson": relations["components_vs_vulnerabilities"]["pearson"],
            "n": relations["components_vs_vulnerabilities"]["n"],
            "fixed_version_available_share": relations[
                "fixed_version_available_share"
            ],
        }
    ]
).to_csv(CSV_DIR / "relations.csv", index=False)
written.append("relations.csv")

for name in ("severity_by_package_type", "findings_by_language"):
    _frame(relations[name], name).to_csv(CSV_DIR / f"{name}.csv", index=False)
    written.append(f"{name}.csv")

print(f"{len(written)} CSV escritos en {CSV_DIR}:")
for name in written:
    print(" -", name)

## 3. Guardar figuras

Figuras canónicas del documento validado, listas para el Visualizer. Se omiten
las que no tienen datos.

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
datasets = document["datasets"]


def _save(fig, name: str):
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return path


figures = []

sev = pd.DataFrame(datasets["severity_distribution"])
if int(sev["count"].sum()) > 0:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(sev["severity"], sev["count"], color="#b2182b")
    ax.set_title("Vulnerabilidades por severidad")
    ax.set_ylabel("Vulnerabilidades")
    figures.append(_save(fig, "severity_distribution.png"))

pkgs = pd.DataFrame(datasets["top_packages"]).head(10)
if not pkgs.empty:
    top = pkgs.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["package"], top["count"], color="#d6604d")
    ax.set_title("Top paquetes por vulnerabilidades")
    ax.set_xlabel("Vulnerabilidades")
    figures.append(_save(fig, "top_packages.png"))

cves = pd.DataFrame(datasets["top_cves"]).head(10)
if not cves.empty:
    top = cves.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["id"], top["count"], color="#4393c3")
    ax.set_title("Top identificadores por apariciones")
    ax.set_xlabel("Apariciones")
    figures.append(_save(fig, "top_cves.png"))

repos = pd.DataFrame(datasets["repository_distribution"])
repos = repos[repos["vulnerabilities"] > 0]
if not repos.empty:
    top = repos.sort_values("vulnerabilities")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["repo"], top["vulnerabilities"], color="#5aae61")
    ax.set_title("Vulnerabilidades por repositorio")
    ax.set_xlabel("Vulnerabilidades")
    figures.append(_save(fig, "repository_distribution.png"))

print(f"{len(figures)} figuras escritas en {FIGURES_DIR}:")
for path in figures:
    print(" -", path.name)

## 4. Observaciones y limitaciones

`observations` son afirmaciones respaldadas por cifras de `coverage`/`datasets`;
`limitations` declara sesgos, secciones ausentes e incertidumbre.

In [ ]:
print("OBSERVACIONES")
for observation in document["observations"]:
    print(
        f"- [{observation['id']}] {observation['title']}: "
        f"{observation['statement']}"
    )

print("\nLIMITACIONES")
for limitation in document["limitations"]:
    print("-", limitation)

## 5. Verificación final

In [ ]:
for path in (OUTPUT_PATH, CSV_DIR, FIGURES_DIR):
    assert path.exists(), f"No existe: {path}"

try:
    shown = OUTPUT_PATH.relative_to(REPO_ROOT)
except ValueError:
    shown = OUTPUT_PATH
print("Documento:", shown, f"({OUTPUT_PATH.stat().st_size} bytes)")
print("CSV      :", len(list(CSV_DIR.glob("*.csv"))), "archivos")
print("Figuras  :", len(list(FIGURES_DIR.glob("*.png"))), "archivos")
print("Validación del contrato: OK")